# Common statistical tests as linear models

Replicates [Lindeløv (2019)](https://lindeloev.github.io/tests-as-linear) using `hea` for the linear-model side and `hea.stats` (a thin scipy wrapper) for the named-test side.

For each row of the cheat sheet:
1. **R-style named test** — `t_test`, `wilcox_test`, `cor_test`, `aov`, `kruskal_test`, `chisq_test`.
2. **Equivalent `lm` / `glm`** — same model, written as a linear regression.
3. **Combined results** — side-by-side comparison.

In [1]:
import numpy as np
import polars as pl

import hea
from hea import (
    aov, chisq_test, cor_test, glm, kruskal_test,
    lm, rank, signed_rank, t_test, wilcox_test,
)
from hea.family import Poisson

# Data is generated by example/stats-data.R (run once with `Rscript`),
# so y/x/y2 here match the post's set.seed(40) draws byte-for-byte.
DATA = 'data/stats'
_main = pl.read_csv(f'{DATA}/main.csv')
y, x, y2 = (_main[c].to_numpy() for c in ('y', 'x', 'y2'))

# Long-format with group indicator (used by sec 7-8).
value = np.r_[y, y2]
group = np.array(['y1'] * 50 + ['y2'] * 50)
group_y2 = (group == 'y2').astype(float)
D2 = pl.DataFrame({'value': value, 'group': group, 'group_y2': group_y2})

## 1. Pearson correlation

In [2]:
a = cor_test(x, y, method='pearson')
print(a)


	Pearson's product-moment correlation

data:  x and y
t = -0.33651, df = 48, p-value = 0.73795
alternative hypothesis: true cor is not equal to 0
95 percent confidence interval:
 -0.32251 0.23298
sample estimates:
cor
-0.048514



In [3]:
Dxy = pl.DataFrame({'x': x, 'y': y})
b = lm('y ~ 1 + x', Dxy)
b.summary()

Formula: y ~ 1 + x

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)  -0.09522     0.25647   -0.6109     0.4204  -0.3713     0.712
x            -0.08718     0.25907   -0.6081     0.4337  -0.3365     0.738
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 50, p = 2, Residual SE = 1.814 on 48 DF
R-Squared = 0.0024, adjusted R-Squared = -0.0184
F-statistics = 0.1132 on 1 and 48 DF, p-value: 0.74

Log Likelihood = -99.6896, AIC = 205.3792, BIC = 211.1153


In [4]:
# r equals the slope when x and y are scaled to SD = 1.
r = a.estimate['cor']
pl.DataFrame({
    'model':     ['cor.test', 'lm'],
    'p.value':   [a.p_value, float(b.p_values['x'].item())],
    't':         [a.statistic['t'], float(b.t_values['x'].item())],
    'r':         [r, float(b.bhat['x'].item())],
    'conf.low':  [a.conf_int[0], float(b.ci_bhat.row(1)[1])],
    'conf.high': [a.conf_int[1], float(b.ci_bhat.row(1)[2])],
})

model,p.value,t,r,conf.low,conf.high
str,f64,f64,f64,f64,f64
"""cor.test""",0.737953,-0.336511,-0.048514,-0.322507,0.23298
"""lm""",0.737953,-0.336511,-0.087181,-0.608082,0.43372


## 2. Spearman correlation

In [5]:
a = cor_test(x, y, method='spearman')
print(a)


	Spearman's rank correlation rho

data:  x and y
S = 21956, p-value = 0.70796
alternative hypothesis: true rho is not equal to 0
sample estimates:
rho
-0.05431



In [6]:
Dr = pl.DataFrame({'rx': rank(x), 'ry': rank(y)})
b = lm('ry ~ 1 + rx', Dr)
b.summary()

Formula: ry ~ 1 + rx

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)  26.88490     4.22287   18.3942    35.3756   6.3665  6.89e-08  ***
rx           -0.05431     0.14412   -0.3441     0.2355  -0.3768     0.708
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 50, p = 2, Residual SE = 14.707 on 48 DF
R-Squared = 0.0029, adjusted R-Squared = -0.0178
F-statistics = 0.1420 on 1 and 48 DF, p-value: 0.71

Log Likelihood = -204.3416, AIC = 414.6831, BIC = 420.4192


In [7]:
# rho and slope are numerically equal: slope of lm(rank(y) ~ rank(x))
# is cov(rx, ry) / var(rx), and var(rx) == var(ry) when there are no ties.
pl.DataFrame({
    'model':    ['cor.test', 'lm'],
    'p.value':  [a.p_value, float(b.p_values['rx'].item())],
    'estimate': [a.estimate['rho'], float(b.bhat['rx'].item())],
})

model,p.value,estimate
str,f64,f64
"""cor.test""",0.707964,-0.05431
"""lm""",0.707964,-0.05431


## 3. One-sample t-test

In [8]:
a = t_test(y)
print(a)


	One Sample t-test

data:  x
t = -0.37466, df = 49, p-value = 0.70953
alternative hypothesis: true mean of x is not equal to 0
95 percent confidence interval:
 -0.60593 0.41549
sample estimates:
mean of x
-0.095216



In [9]:
Dy = pl.DataFrame({'y': y})
b = lm('y ~ 1', Dy)
b.summary()

Formula: y ~ 1

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)  -0.09522     0.25414   -0.6059     0.4155  -0.3747     0.710
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 50, p = 1, Residual SE = 1.797 on 49 DF
R-Squared = -0.0000, adjusted R-Squared = -0.0000
Log Likelihood = -99.7485, AIC = 203.4971, BIC = 207.3211


In [10]:
pl.DataFrame({
    'model':     ['t.test', 'lm'],
    'mean':      [a.estimate['mean of x'], float(b.bhat['(Intercept)'].item())],
    'p.value':   [a.p_value, float(b.p_values['(Intercept)'].item())],
    't':         [a.statistic['t'], float(b.t_values['(Intercept)'].item())],
    'df':        [a.parameter['df'], b.df_residuals],
    'conf.low':  [a.conf_int[0], float(b.ci_bhat.row(0)[1])],
    'conf.high': [a.conf_int[1], float(b.ci_bhat.row(0)[2])],
})

model,mean,p.value,t,df,conf.low,conf.high
str,f64,f64,f64,f64,f64,f64
"""t.test""",-0.095216,0.709528,-0.374662,49.0,-0.605925,0.415493
"""lm""",-0.095216,0.709528,-0.374662,49.0,-0.605925,0.415493


## 4. Wilcoxon signed-rank

In [11]:
a = wilcox_test(y)
print(a)


	Wilcoxon signed rank test with continuity correction

data:  x
V = 508, p-value = 0.21505
alternative hypothesis: true value is not equal to 0



In [12]:
Dsr = pl.DataFrame({'sr': signed_rank(y)})
b = lm('sr ~ 1', Dsr)
b.summary()

Formula: sr ~ 1

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)    -5.180       4.120   -13.459      3.099   -1.257     0.215
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 50, p = 1, Residual SE = 29.131 on 49 DF
R-Squared = 0.0000, adjusted R-Squared = 0.0000
Log Likelihood = -239.0327, AIC = 482.0655, BIC = 485.8895


In [13]:
# Bonus: lm on signed ranks IS a one-sample t-test on signed ranks.
c = t_test(signed_rank(y))
pl.DataFrame({
    'model':     ['wilcox.test', 'lm(signed_rank)', 't.test(signed_rank)'],
    'p.value':   [a.p_value, float(b.p_values['(Intercept)'].item()), c.p_value],
    'mean_rank': [None, float(b.bhat['(Intercept)'].item()), c.estimate['mean of x']],
})

model,p.value,mean_rank
str,f64,f64
"""wilcox.test""",0.215052,null
"""lm(signed_rank)""",0.21459,-5.18
"""t.test(signed_rank)""",0.21459,-5.18


## 5. Paired-sample t-test

In [14]:
a = t_test(y, y2, paired=True)
print(a)


	Paired t-test

data:  x and y
t = -1.7108, df = 49, p-value = 0.093448
alternative hypothesis: true mean of the differences is not equal to 0
95 percent confidence interval:
 -1.2944 0.10396
sample estimates:
mean of the differences
-0.59522



In [15]:
Dd = pl.DataFrame({'d': y - y2})
b = lm('d ~ 1', Dd)
b.summary()

Formula: d ~ 1

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)   -0.5952      0.3479   -1.2944     0.1040   -1.711    0.0934  .
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 50, p = 1, Residual SE = 2.460 on 49 DF
R-Squared = 0.0000, adjusted R-Squared = 0.0000
Log Likelihood = -115.4537, AIC = 234.9074, BIC = 238.7314


In [16]:
pl.DataFrame({
    'model':     ['t.test', 'lm'],
    'mean':      [a.estimate['mean of the differences'], float(b.bhat['(Intercept)'].item())],
    'p.value':   [a.p_value, float(b.p_values['(Intercept)'].item())],
    'df':        [a.parameter['df'], b.df_residuals],
    't':         [a.statistic['t'], float(b.t_values['(Intercept)'].item())],
    'conf.low':  [a.conf_int[0], float(b.ci_bhat.row(0)[1])],
    'conf.high': [a.conf_int[1], float(b.ci_bhat.row(0)[2])],
})

model,mean,p.value,df,t,conf.low,conf.high
str,f64,f64,f64,f64,f64,f64
"""t.test""",-0.595216,0.093448,49.0,-1.710771,-1.294393,0.103961
"""lm""",-0.595216,0.093448,49.0,-1.710771,-1.294393,0.103961


## 6. Wilcoxon matched pairs

In [17]:
a = wilcox_test(y, y2, paired=True)
print(a)


	Wilcoxon signed rank test with continuity correction

data:  x and y
V = 429, p-value = 0.043975
alternative hypothesis: true value is not equal to 0



In [18]:
Dd = pl.DataFrame({'sr_d': signed_rank(y - y2)})
b = lm('sr_d ~ 1', Dd)
b.summary()

Formula: sr_d ~ 1

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)    -8.340       4.013  -16.4036    -0.2764   -2.078    0.0429  *
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 50, p = 1, Residual SE = 28.373 on 49 DF
R-Squared = 0.0000, adjusted R-Squared = 0.0000
Log Likelihood = -237.7143, AIC = 479.4286, BIC = 483.2527


In [19]:
# Bonus: lm on signed_rank(y - y2) IS t.test on signed_rank(y - y2).
c = t_test(signed_rank(y - y2))
pl.DataFrame({
    'model':          ['wilcox.test', 'lm(signed_rank)', 't.test(signed_rank)'],
    'p.value':        [a.p_value, float(b.p_values['(Intercept)'].item()), c.p_value],
    'mean_rank_diff': [None, float(b.bhat['(Intercept)'].item()), c.estimate['mean of x']],
})

model,p.value,mean_rank_diff
str,f64,f64
"""wilcox.test""",0.043975,null
"""lm(signed_rank)""",0.042926,-8.34
"""t.test(signed_rank)""",0.042926,-8.34


## 7. Independent (two-sample) t-test

In [20]:
a = t_test(y, y2, var_equal=True)
print(a)


	Two Sample t-test

data:  x and y
t = -1.798, df = 98, p-value = 0.075251
alternative hypothesis: true mean of x is not equal to 0
95 percent confidence interval:
 -1.2521 0.061718
sample estimates:
mean of x  mean of y
-0.095216  0.5



In [21]:
b = lm('value ~ 1 + group_y2', D2)
b.summary()

Formula: value ~ 1 + group_y2

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)  -0.09522     0.23408  -0.55974    0.36931  -0.4068    0.6851
group_y2      0.59522     0.33104  -0.06172    1.25215   1.7980    0.0753  .
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 100, p = 2, Residual SE = 1.655 on 98 DF
R-Squared = 0.0319, adjusted R-Squared = 0.0221
F-statistics = 3.2329 on 1 and 98 DF, p-value: 0.08

Log Likelihood = -191.2753, AIC = 388.5505, BIC = 396.3660


In [22]:
pl.DataFrame({
    'model':       ['t.test', 'lm'],
    'mean_y':      [a.estimate['mean of x'], float(b.bhat['(Intercept)'].item())],
    'difference':  [a.estimate['mean of y'] - a.estimate['mean of x'], float(b.bhat['group_y2'].item())],
    'p.value':     [a.p_value, float(b.p_values['group_y2'].item())],
    'df':          [a.parameter['df'], b.df_residuals],
    't':           [a.statistic['t'], float(b.t_values['group_y2'].item())],
})

model,mean_y,difference,p.value,df,t
str,f64,f64,f64,f64,f64
"""t.test""",-0.095216,0.595216,0.075251,98.0,-1.798029
"""lm""",-0.095216,0.595216,0.075251,98.0,1.798029


## 8. Mann-Whitney U

In [23]:
a = wilcox_test(y, y2)
print(a)


	Wilcoxon rank sum test with continuity correction

data:  x and y
W = 924, p-value = 0.024836
alternative hypothesis: true value is not equal to 0



In [24]:
Drv = D2.with_columns(rank_value=pl.Series(rank(value)))
b = lm('rank_value ~ 1 + group_y2', Drv)
b.summary()

Formula: rank_value ~ 1 + group_y2

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)    43.980       4.017    36.008     51.952   10.948    <2e-16  ***
group_y2       13.040       5.681     1.766     24.314    2.295    0.0238  *
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 100, p = 2, Residual SE = 28.406 on 98 DF
R-Squared = 0.0510, adjusted R-Squared = 0.0413
F-statistics = 5.2685 on 1 and 98 DF, p-value: 0.02

Log Likelihood = -475.5423, AIC = 957.0846, BIC = 964.9001


In [25]:
pl.DataFrame({
    'model':     ['wilcox.test', 'lm(rank)'],
    'p.value':   [a.p_value, float(b.p_values['group_y2'].item())],
    'rank_diff': [None, float(b.bhat['group_y2'].item())],
})

model,p.value,rank_diff
str,f64,f64
"""wilcox.test""",0.024836,null
"""lm(rank)""",0.023846,13.04


## 9. One-way ANOVA

In [26]:
# ANOVA / Kruskal / ANCOVA / two-way data, also from example/stats-data.R.
Da = pl.read_csv(f'{DATA}/anova.csv')

In [27]:
a = aov('value ~ group', Da)
print(a)

Anova Table (Type II tests)

Response: value
                Sum Sq  Df   F value      Pr(>F)
group               10   2         5   0.0099839
Residuals           57  57


In [28]:
b = lm('value ~ 1 + group_b + group_c', Da)
b.summary()

Formula: value ~ 1 + group_b + group_c

Coefficients:
              Estimate  Std. Error  CI[2.5%]  CI[97.5]%    t value  Pr(>|t|)
(Intercept)  7.262e-16      0.2236   -0.4478     0.4478  3.248e-15   1.00000
group_b         1.0000      0.3162    0.3668     1.6332      3.162   0.00251  **
group_c         0.5000      0.3162   -0.1332     1.1332      1.581   0.11938
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 60, p = 3, Residual SE = 1.000 on 57 DF
R-Squared = 0.1493, adjusted R-Squared = 0.1194
F-statistics = 5.0000 on 2 and 57 DF, p-value: 0.01

Log Likelihood = -83.5975, AIC = 175.1950, BIC = 183.5724


In [29]:
row = a.rows[0]
pl.DataFrame({
    'model':       ['Anova', 'lm'],
    'df':          [row['df'], b.df_model],
    'df.residual': [a.residual_df, b.df_residuals],
    'F':           [row['F'], b.fstats],
    'p.value':     [row['p'], b.f_p_value],
})

model,df,df.residual,F,p.value
str,i64,i64,f64,f64
"""Anova""",2,57,5.0,0.009984
"""lm""",2,57,5.0,0.009984


## 10. Kruskal-Wallis

In [30]:
a = kruskal_test('value ~ group', Da)
print(a)


	Kruskal-Wallis rank sum test

data:  value by group
Kruskal-Wallis chi-squared = 8.7846, df = 2, p-value = 0.012372



In [31]:
Dar = Da.with_columns(rank_value=pl.Series(rank(Da['value'].to_numpy())))
b = lm('rank_value ~ 1 + group_b + group_c', Dar)
b.summary()

Formula: rank_value ~ 1 + group_b + group_c

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)    22.100       3.665    14.760     29.440    6.029  1.29e-07  ***
group_b        16.350       5.184     5.970     26.730    3.154   0.00257  **
group_c         8.850       5.184    -1.530     19.230    1.707   0.09321  .
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 60, p = 3, Residual SE = 16.392 on 57 DF
R-Squared = 0.1489, adjusted R-Squared = 0.1190
F-statistics = 4.9857 on 2 and 57 DF, p-value: 0.01

Log Likelihood = -251.4050, AIC = 510.8100, BIC = 519.1874


In [32]:
pl.DataFrame({
    'model':   ['kruskal.test', 'lm(rank)'],
    'df':      [a.parameter['df'], b.df_model],
    'p.value': [a.p_value, b.f_p_value],
})

model,df,p.value
str,i64,f64
"""kruskal.test""",2,0.012372
"""lm(rank)""",2,0.010106


## 11. One-way ANCOVA

In [33]:
# Da already has `age` (correlated with value).
Dc = Da

In [34]:
a = aov('value ~ group + age', Dc)
print(a)

Anova Table (Type II tests)

Response: value
                Sum Sq  Df   F value      Pr(>F)
group           6.4649   2    3.7103    0.030677
age             8.2125   1    9.4265   0.0032961
Residuals       48.788  56


In [35]:
b = lm('value ~ 1 + group_b + group_c + age', Dc)
b.summary()

Formula: value ~ 1 + group_b + group_c + age

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  t value  Pr(>|t|)
(Intercept)   0.03800     0.20908  -0.38083    0.45683   0.1817   0.85644
group_b       0.81950     0.30096   0.21661    1.42240   2.7230   0.00861  **
group_c       0.39280     0.29722  -0.20260    0.98821   1.3216   0.19168
age           0.11580     0.03772   0.04024    0.19136   3.0703   0.00330  **
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 60, p = 4, Residual SE = 0.933 on 56 DF
R-Squared = 0.2718, adjusted R-Squared = 0.2328
F-statistics = 6.9683 on 3 and 56 DF, p-value: 0.00

Log Likelihood = -78.9302, AIC = 167.8604, BIC = 178.3322


In [36]:
# Match the post: Anova table side-by-side with lm-LRT for each term.
from hea import anova

full = lm('value ~ 1 + group_b + group_c + age', Dc)
null_age   = lm('value ~ 1 + group_b + group_c', Dc)
null_group = lm('value ~ 1 + age', Dc)

pl.DataFrame({
    'term':    ['age', 'age',   'group', 'group'],
    'model':   ['Anova', 'lm',  'Anova', 'lm'],
    'df':      [r['df'] for r in [a.rows[1], a.rows[1], a.rows[0], a.rows[0]]],
    'F':       [
        a.rows[1]['F'],
        ((null_age.rss - full.rss) / 1) / (full.rss / full.df_residuals),
        a.rows[0]['F'],
        ((null_group.rss - full.rss) / 2) / (full.rss / full.df_residuals),
    ],
    'p.value': [a.rows[1]['p'], a.rows[1]['p'], a.rows[0]['p'], a.rows[0]['p']],
})

term,model,df,F,p.value
str,str,i64,f64,f64
"""age""","""Anova""",1,9.426539,0.003296
"""age""","""lm""",1,9.426539,0.003296
"""group""","""Anova""",2,3.7103,0.030677
"""group""","""lm""",2,3.7103,0.030677


## 12. Two-way ANOVA

In [37]:
# Da already has `mood` and `mood_happy`. Just add the interaction columns.
Dm = Da.with_columns(
    group_b_mood_happy=pl.col('group_b') * pl.col('mood_happy'),
    group_c_mood_happy=pl.col('group_c') * pl.col('mood_happy'),
)

In [38]:
a = aov('value ~ mood * group', Dm)
print(a)

Anova Table (Type II tests)

Response: value
                Sum Sq  Df   F value      Pr(>F)
mood                 0   0                      
group       -7.1054e-15   0                      
mood:group      5.1759   2    2.9014    0.063554
Residuals       48.166  54


In [39]:
# Test the interaction via LRT: full vs no-interaction.
full = lm(
    'value ~ 1 + group_b + group_c + mood_happy '
    '+ group_b_mood_happy + group_c_mood_happy',
    Dm,
)
null = lm('value ~ 1 + group_b + group_c + mood_happy', Dm)
full.summary()

Formula: value ~ 1 + group_b + group_c + mood_happy + group_b_mood_happy + group_c_mood_happy

Coefficients:
                    Estimate  Std. Error  CI[2.5%]  CI[97.5]%   t value  Pr(>|t|)
(Intercept)          0.01930     0.29866  -0.57947    0.61807   0.06464    0.9487
group_b              0.87676     0.42236   0.02997    1.72355   2.07584    0.0427  *
group_c             -0.17546     0.42236  -1.02225    0.67133  -0.41542    0.6795
mood_happy          -0.03861     0.42236  -0.88540    0.80818  -0.09141    0.9275
group_b_mood_happy   0.24648     0.59731  -0.95106    1.44402   0.41265    0.6815
group_c_mood_happy   1.35092     0.59731   0.15338    2.54846   2.26166    0.0278  *
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

n = 60, p = 6, Residual SE = 0.944 on 54 DF
R-Squared = 0.2811, adjusted R-Squared = 0.2145
F-statistics = 4.2231 on 5 and 54 DF, p-value: 0.00

Log Likelihood = -78.5454, AIC = 171.0908, BIC = 185.7512


In [40]:
from scipy.stats import f as _f

interaction = next(r for r in a.rows if 'mood' in r['term'] and 'group' in r['term'])
df_inter = full.df_model - null.df_model
F_lrt = ((null.rss - full.rss) / df_inter) / (full.rss / full.df_residuals)
pl.DataFrame({
    'model':     ['Anova mood:group', 'lm LRT'],
    'F':         [interaction['F'], F_lrt],
    'df':        [interaction['df'], df_inter],
    'p.value':   [interaction['p'], float(_f.sf(F_lrt, df_inter, full.df_residuals))],
    'sumsq':     [interaction['sum_sq'], null.rss - full.rss],
    'res.sumsq': [a.residual_ss, full.rss],
})

model,F,df,p.value,sumsq,res.sumsq
str,f64,i64,f64,f64,f64
"""Anova mood:group""",2.901414,2,0.063554,5.175874,48.165693
"""lm LRT""",2.901414,2,0.063554,5.175874,48.165693


## 13. Goodness of fit (chi-square)

In [41]:
Dg = pl.DataFrame({
    'mood':       ['happy', 'sad', 'meh'],
    'counts':     [60.0, 90.0, 70.0],
    'mood_happy': [1.0, 0.0, 0.0],
    'mood_sad':   [0.0, 1.0, 0.0],
})

In [42]:
a = chisq_test(Dg['counts'].to_numpy())
print(a)


	Chi-squared test for given probabilities

data:  x
X-squared = 6.3636, df = 2, p-value = 0.04151



In [43]:
full = glm('counts ~ 1 + mood_happy + mood_sad', Dg, family=Poisson())
null = glm('counts ~ 1', Dg, family=Poisson())
full.summary()

Call:
glm(formula = counts ~ 1 + mood_happy + mood_sad, family = poisson(link=log))

Coefficients:
             Estimate  Std. Error  CI[2.5%]  CI[97.5]%  z value  Pr(>|z|)
(Intercept)    4.2485      0.1195   4.01423    4.48276  35.5455    <2e-16  ***
mood_happy    -0.1542      0.1759  -0.49897    0.19067  -0.8762     0.381
mood_sad       0.2513      0.1594  -0.06103    0.56366   1.5770     0.115
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for poisson family taken to be 1; fixed)

    Null deviance: 6.2697  on 2 degrees of freedom
Residual deviance: -0.0000  on 0 degrees of freedom
AIC: 24.3633    BIC: 21.6591    logLik: -9.1816

Number of Fisher Scoring iterations: 2

In [44]:
from scipy.stats import chi2 as _chi2

# Rao score statistic for the goodness-of-fit GLM equals chisq.test exactly.
n = Dg['counts'].sum()
expected_null = n / Dg.height
rao = float(np.sum((Dg['counts'].to_numpy() - expected_null) ** 2 / expected_null))
df_full_vs_null = null.df_residual - full.df_residual
lrt = float(null.deviance - full.deviance)
pl.DataFrame({
    'model':   ['chisq.test', 'glm Rao', 'glm LRT'],
    'Chisq':   [a.statistic['X-squared'], rao, lrt],
    'df':      [a.parameter['df'], df_full_vs_null, df_full_vs_null],
    'p.value': [
        a.p_value,
        float(_chi2.sf(rao, df_full_vs_null)),
        float(_chi2.sf(lrt, df_full_vs_null)),
    ],
})

model,Chisq,df,p.value
str,f64,i64,f64
"""chisq.test""",6.363636,2,0.04151
"""glm Rao""",6.363636,2,0.04151
"""glm LRT""",6.269709,2,0.043506


## 14. Contingency table (chi-square)

In [45]:
Dc = pl.DataFrame({
    'mood': ['happy', 'happy', 'meh', 'meh', 'sad', 'sad'],
    'sex':  ['male', 'female', 'male', 'female', 'male', 'female'],
    'Freq': [100.0, 70.0, 30.0, 32.0, 110.0, 120.0],
}).with_columns(
    mood_happy=(pl.col('mood') == 'happy').cast(pl.Float64),
    mood_meh=(pl.col('mood') == 'meh').cast(pl.Float64),
    sex_male=(pl.col('sex') == 'male').cast(pl.Float64),
).with_columns(
    mood_happy_sex_male=pl.col('mood_happy') * pl.col('sex_male'),
    mood_meh_sex_male=pl.col('mood_meh') * pl.col('sex_male'),
)
Dc_table = (
    Dc.pivot(values='Freq', index='sex', on='mood').drop('sex').to_numpy()
)

In [46]:
a = chisq_test(Dc_table)
print(a)


	Pearson's Chi-squared test

data:  x
X-squared = 5.0999, df = 2, p-value = 0.078087



In [47]:
full = glm(
    'Freq ~ 1 + mood_happy + mood_meh + sex_male '
    '+ mood_happy_sex_male + mood_meh_sex_male',
    Dc, family=Poisson(),
)
null = glm('Freq ~ 1 + mood_happy + mood_meh + sex_male', Dc, family=Poisson())
full.summary()

Call:
glm(formula = Freq ~ 1 + mood_happy + mood_meh + sex_male + mood_happy_sex_male + mood_meh_sex_male, family = poisson(link=log))

Coefficients:
                     Estimate  Std. Error  CI[2.5%]  CI[97.5]%   z value  Pr(>|z|)
(Intercept)           4.78749     0.09129   4.60857    4.96641  52.44434   < 2e-16  ***
mood_happy           -0.53900     0.15040  -0.83377   -0.24423  -3.58384  0.000339  ***
mood_meh             -1.32176     0.19896  -1.71170   -0.93181  -6.64347  3.06e-11  ***
sex_male             -0.08701     0.13200  -0.34573    0.17171  -0.65917  0.509785
mood_happy_sex_male   0.44369     0.20423   0.04340    0.84397   2.17248  0.029819  *
mood_meh_sex_male     0.02247     0.28637  -0.53880    0.58375   0.07847  0.937450
---
Signif. codes: 0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

(Dispersion parameter for poisson family taken to be 1; fixed)

    Null deviance: 111.1298  on 5 degrees of freedom
Residual deviance: -0.0000  on 0 degrees of freedom
AIC: 48.2535   

In [48]:
from scipy.stats import chi2 as _chi2

# Rao score statistic on the saturated null gives Pearson's X² exactly.
expected = (Dc_table.sum(axis=1, keepdims=True)
            * Dc_table.sum(axis=0, keepdims=True) / Dc_table.sum())
rao = float(np.sum((Dc_table - expected) ** 2 / expected))
df_inter = null.df_residual - full.df_residual
lrt = float(null.deviance - full.deviance)
pl.DataFrame({
    'model':   ['chisq.test', 'glm Rao', 'glm LRT'],
    'Chisq':   [a.statistic['X-squared'], rao, lrt],
    'df':      [a.parameter['df'], df_inter, df_inter],
    'p.value': [
        a.p_value,
        float(_chi2.sf(rao, df_inter)),
        float(_chi2.sf(lrt, df_inter)),
    ],
})

model,Chisq,df,p.value
str,f64,i64,f64
"""chisq.test""",5.099859,2,0.078087
"""glm Rao""",5.099859,2,0.078087
"""glm LRT""",5.119915,2,0.077308
